In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import (
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_score,
)


# =========================================================
# EXPERIMENTS
# 特徴ベクトル抽出時と同じ exp_name を使う
# =========================================================
EXPERIMENTS = [
    # ======================================================
    # 1. ViT CLS / AVG
    # ======================================================
    {
        "exp_name": "vit_large_patch16_384_augreg_CLS_384",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "img_size": 384,
        "feature_type": "cls",
    },
    {
        "exp_name": "vit_large_patch16_384_augreg_CLS_616",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "img_size": 616,
        "feature_type": "cls",
    },
    {
        "exp_name": "vit_large_patch16_384_augreg_AVG_384",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "img_size": 384,
        "feature_type": "avg",
    },
    {
        "exp_name": "vit_large_patch16_384_augreg_AVG_616",
        "model_name": "vit_large_patch16_384.augreg_in21k_ft_in1k",
        "img_size": 616,
        "feature_type": "avg",
    },

    # ======================================================
    # 2. EVA02 AVG
    # ======================================================
    {
        "exp_name": "eva02_large_patch14_448_mim_in22k_ft_in22k_AVG_448",
        "model_name": "eva02_large_patch14_448.mim_in22k_ft_in22k",
        "img_size": 448,
        "feature_type": "avg",
    },
    {
        "exp_name": "eva02_large_patch14_448_mim_in22k_ft_in22k_in1k_AVG_448",
        "model_name": "eva02_large_patch14_448.mim_in22k_ft_in22k_in1k",
        "img_size": 448,
        "feature_type": "avg",
    },
    {
        "exp_name": "eva02_large_patch14_448_mim_m38m_ft_in22k_AVG_448",
        "model_name": "eva02_large_patch14_448.mim_m38m_ft_in22k",
        "img_size": 448,
        "feature_type": "avg",
    },
    {
        "exp_name": "eva02_large_patch14_448_mim_m38m_ft_in22k_in1k_AVG_448",
        "model_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
        "img_size": 448,
        "feature_type": "avg",
    },

    # ======================================================
    # 3. DINO / SigLIP
    # ======================================================
    {
        "exp_name": "vit_large_patch14_dinov2_AVG_518",
        "model_name": "vit_large_patch14_dinov2.lvd142m",
        "img_size": 518,
        "feature_type": "avg",
    },
    {
        "exp_name": "vit_large_patch16_siglip_gap_512_AVG_512",
        "model_name": "vit_large_patch16_siglip_gap_512.v2_webli",
        "img_size": 512,
        "feature_type": "avg",
    },

    # ======================================================
    # 4. ConvNet / CNN系
    # ======================================================
    {
        "exp_name": "tf_efficientnetv2_l_AVG_480",
        "model_name": "tf_efficientnetv2_l.in21k_ft_in1k",
        "img_size": 480,
        "feature_type": "avg",
    },
    {
        "exp_name": "convnextv2_large_fcmae_AVG_384",
        "model_name": "convnextv2_large.fcmae_ft_in22k_in1k_384",
        "img_size": 384,
        "feature_type": "avg",
    },

    # ======================================================
    # 5. Swin / Hybrid / MetaFormer
    # ======================================================
    {
        "exp_name": "swinv2_large_window12to24_AVG_384",
        "model_name": "swinv2_large_window12to24_192to384.ms_in22k_ft_in1k",
        "img_size": 384,
        "feature_type": "avg",
    },
    {
        "exp_name": "caformer_b36_AVG_384",
        "model_name": "caformer_b36.sail_in22k_ft_in1k_384",
        "img_size": 384,
        "feature_type": "avg",
    },
    {
        "exp_name": "maxxvitv2_rmlp_base_AVG_384",
        "model_name": "maxxvitv2_rmlp_base_rw_384.sw_in12k_ft_in1k",
        "img_size": 384,
        "feature_type": "avg",
    },
]


# =========================================================
# 全体設定
# =========================================================
FEATURE_BASE_1 = Path("/home/tatsushi/デスクトップ/sensitivity_data7/image-feature")
FEATURE_BASE_2 = Path("/home/tatsushi/デスクトップ/胃癌取り扱い規約trimming/image-feature")

RESULT_BASE = Path("/home/tatsushi/デスクトップ/val-result_weight")

folder_name = input("保存フォルダー名 > ").strip()
if folder_name == "":
    folder_name = "all_experiments"

NUM_SEEDS = int(input("seed数を入力してください。例: 40 > ").strip() or "40")

BATCH_SIZE = 32
NUM_EPOCHS = 20
PATIENCE = 5
TOP_K_ENSEMBLE = 5
NUM_WORKERS = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABEL_MAPPING = {
    "2_resis": 0,
    "0_sens": 1,
}


# =========================================================
# 乱数固定
# =========================================================
def set_global_seed(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


def seed_worker(worker_id: int, seed: int) -> None:
    worker_seed = seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)


# =========================================================
# パス作成
# =========================================================
def build_train_roots(exp_name: str):
    return [
        (FEATURE_BASE_1 / exp_name / "mucosa/labeled", 1.0),
        (FEATURE_BASE_1 / exp_name / "mucosa/unlabeled", 0.9),
        (FEATURE_BASE_1 / exp_name / "mucosa/detekita", 1.0),
        (FEATURE_BASE_1 / exp_name / "mucosa/unknown", 0.9),
        (FEATURE_BASE_1 / exp_name / "mucosa/irregular", 0.9),
        (FEATURE_BASE_1 / exp_name / "002_classes", 0.9),
        (FEATURE_BASE_1 / exp_name / "web_jpg", 1.0),
        (FEATURE_BASE_1 / exp_name / "web_99", 1.0),
        (FEATURE_BASE_2 / exp_name, 0.9),
    ]


def build_val_roots(exp_name: str):
    return [
        FEATURE_BASE_1 / exp_name / "mucosa/sub_external",
    ]


def build_external_root(exp_name: str):
    return FEATURE_BASE_1 / exp_name / "mucosa/external"


def make_output_paths(exp_name: str):
    out_root = RESULT_BASE / folder_name / exp_name
    model_dir = out_root / "Vit_fixed_val_classes_models_0sens_positive"
    external_dir = out_root / "external_eval"

    model_dir.mkdir(parents=True, exist_ok=True)
    external_dir.mkdir(parents=True, exist_ok=True)

    return {
        "out_root": out_root,
        "model_dir": model_dir,
        "external_dir": external_dir,
        "train_xlsx": out_root / "Vit_fixed_val_classes_top5_ensemble_0sens_positive.xlsx",
        "best_model": model_dir / "best_overall_model.pth",
        "ensemble_info": model_dir / "top5_ensemble_info.pth",
        "external_xlsx": external_dir / "external_eval_result_0sens_positive.xlsx",
        "external_csv": external_dir / "external_eval_summary_0sens_positive.csv",
    }


# =========================================================
# torch.load
# =========================================================
def torch_load_safe(path, map_location="cpu"):
    return torch.load(path, map_location=map_location, weights_only=False)


# =========================================================
# サンプル収集
# =========================================================
def collect_samples_from_roots(root_dirs, with_weight=True):
    samples = []

    if isinstance(root_dirs, (str, Path)):
        root_dirs = [(root_dirs, 1.0)]

    for root_item in root_dirs:
        if isinstance(root_item, dict):
            root_dir = root_item["path"]
            root_weight = float(root_item.get("weight", 1.0))
        elif isinstance(root_item, (tuple, list)):
            root_dir = root_item[0]
            root_weight = float(root_item[1]) if len(root_item) > 1 else 1.0
        else:
            root_dir = root_item
            root_weight = 1.0

        root_dir = Path(root_dir)
        if not root_dir.exists():
            print(f"[SKIP] root not found: {root_dir}")
            continue

        for cls_name, label in LABEL_MAPPING.items():
            class_dir = root_dir / cls_name
            if not class_dir.exists():
                continue

            for sub_dir in class_dir.iterdir():
                if not sub_dir.is_dir():
                    continue

                case_id = f"{cls_name}/{sub_dir.name}"

                for pt_path in sub_dir.rglob("*.pt"):
                    item = {
                        "path": str(pt_path),
                        "label": label,
                        "class_name": cls_name,
                        "subfolder": sub_dir.name,
                        "case_id": case_id,
                        "root": str(root_dir),
                    }
                    if with_weight:
                        item["sample_weight"] = root_weight

                    samples.append(item)

    return samples


def collect_samples_from_root(root_dir):
    return collect_samples_from_roots([root_dir], with_weight=False)


def get_case_ids(samples):
    return set(s["case_id"] for s in samples)


# =========================================================
# 特徴ベクトル次元を自動判定
# =========================================================
def load_feature_tensor(path):
    feat = torch_load_safe(path, map_location="cpu")

    if isinstance(feat, np.ndarray):
        feat = torch.from_numpy(feat)

    feat = feat.float()

    # [1, dim] や [1, 1, dim] などを吸収
    feat = feat.squeeze()

    # もし [C,H,W] などが来ても MLP入力用に1次元化
    if feat.ndim > 1:
        feat = feat.flatten()

    return feat


def infer_input_dim(samples):
    if len(samples) == 0:
        raise ValueError("samples が0件のため INPUT_DIM を推定できません。")

    feat = load_feature_tensor(samples[0]["path"])
    return int(feat.numel())


# =========================================================
# Dataset
# =========================================================
class FeatureDataset(Dataset):
    def __init__(self, samples, return_weight: bool = False, return_meta: bool = False):
        self.samples = samples
        self.return_weight = return_weight
        self.return_meta = return_meta

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        feat = load_feature_tensor(item["path"])
        label = int(item["label"])

        if self.return_meta:
            return {
                "feat": feat,
                "label": label,
                "case_id": item["case_id"],
                "path": item["path"],
                "class_name": item["class_name"],
                "subfolder": item["subfolder"],
                "root": item["root"],
            }

        if self.return_weight:
            weight = torch.tensor(item.get("sample_weight", 1.0), dtype=torch.float32)
            return feat, label, weight

        return feat, label


# =========================================================
# MLP
# =========================================================
class MLPClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 256, num_classes: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# =========================================================
# 学習
# =========================================================
def train_and_evaluate(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=40,
    patience=5,
):
    best_wts = None
    best_epoch = 0
    best_val_acc = 0.0
    best_val_loss = float("inf")
    patience_cnt = 0
    history = []

    for epoch in range(1, num_epochs + 1):
        model.train()

        train_loss_sum = 0.0
        train_weight_sum = 0.0
        train_corr = 0
        train_total = 0

        for feats, labels, weights in train_loader:
            feats = feats.to(device)
            labels = labels.to(device)
            weights = weights.to(device)

            optimizer.zero_grad()

            out = model(feats)
            sample_losses = criterion(out, labels)
            loss = (sample_losses * weights).sum() / weights.sum().clamp_min(1e-8)

            preds = out.argmax(1)

            loss.backward()
            optimizer.step()

            train_loss_sum += (sample_losses.detach() * weights).sum().item()
            train_weight_sum += weights.sum().item()
            train_corr += (preds == labels).sum().item()
            train_total += feats.size(0)

        train_loss = train_loss_sum / max(train_weight_sum, 1e-8)
        train_acc = train_corr / max(train_total, 1)

        model.eval()

        val_loss_sum = 0.0
        val_corr = 0
        val_total = 0

        with torch.no_grad():
            for feats, labels in val_loader:
                feats = feats.to(device)
                labels = labels.to(device)

                out = model(feats)
                sample_losses = criterion(out, labels)
                preds = out.argmax(1)

                val_loss_sum += sample_losses.sum().item()
                val_corr += (preds == labels).sum().item()
                val_total += feats.size(0)

        val_loss = val_loss_sum / max(val_total, 1)
        val_acc = val_corr / max(val_total, 1)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch
            patience_cnt = 0
            best_wts = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            patience_cnt += 1

        if patience_cnt >= patience:
            break

    if best_wts is not None:
        model.load_state_dict(best_wts)

    return model, best_epoch, best_val_acc, best_val_loss, history


# =========================================================
# 予測
# =========================================================
def collect_preds_from_loader(model, loader, device):
    y_true_all = []
    y_proba_all = []

    model.eval()

    with torch.no_grad():
        for feats, labels in loader:
            feats = feats.to(device)

            out = model(feats)
            proba = torch.softmax(out, dim=1)[:, 1]

            y_true_all.extend(labels.numpy().tolist())
            y_proba_all.extend(proba.cpu().numpy().tolist())

    return np.array(y_true_all), np.array(y_proba_all)


def collect_ensemble_preds_from_loader(models, loader, device):
    y_true_all = []
    y_proba_all = []

    for m in models:
        m.eval()

    with torch.no_grad():
        for feats, labels in loader:
            feats = feats.to(device)

            prob_list = []
            for model in models:
                out = model(feats)
                proba = torch.softmax(out, dim=1)[:, 1]
                prob_list.append(proba.unsqueeze(1))

            prob_stack = torch.cat(prob_list, dim=1)
            mean_prob = prob_stack.mean(dim=1)

            y_true_all.extend(labels.numpy().tolist())
            y_proba_all.extend(mean_prob.cpu().numpy().tolist())

    return np.array(y_true_all), np.array(y_proba_all)


# =========================================================
# metrics
# =========================================================
def calc_binary_metrics_0sens_positive(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    recall_0sens = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    precision_0sens = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_0sens = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    specificity_0sens = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "accuracy": acc,
        "recall_0_sens": recall_0sens,
        "specificity_0_sens": specificity_0sens,
        "precision_0_sens": precision_0sens,
        "f1_0_sens": f1_0sens,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def get_metrics_with_best_cutoff(y_true, y_proba):
    best_thr = 0.5
    best_score = -1.0
    best_metrics = None

    for thr in np.arange(0.2, 0.81, 0.05):
        y_pred = (y_proba >= thr).astype(int)

        metrics = calc_binary_metrics_0sens_positive(y_true, y_pred)
        score = metrics["f1_0_sens"]

        if score > best_score:
            best_score = score
            best_thr = thr
            best_metrics = metrics

    return (
        best_thr,
        best_metrics["accuracy"],
        best_metrics["recall_0_sens"],
        best_metrics["specificity_0_sens"],
        best_metrics["f1_0_sens"],
        best_score,
        best_metrics["precision_0_sens"],
        best_metrics["tn"],
        best_metrics["fp"],
        best_metrics["fn"],
        best_metrics["tp"],
    )


def calc_metrics(y_true, y_pred):
    m = calc_binary_metrics_0sens_positive(y_true, y_pred)
    return {
        "acc": m["accuracy"],
        "recall_0_sens": m["recall_0_sens"],
        "specificity_0_sens": m["specificity_0_sens"],
        "precision_0_sens": m["precision_0_sens"],
        "f1_0_sens": m["f1_0_sens"],
        "tn": m["tn"],
        "fp": m["fp"],
        "fn": m["fn"],
        "tp": m["tp"],
    }


# =========================================================
# 学習＋Top5 ensemble
# =========================================================
def run_training_for_experiment(exp):
    exp_name = exp["exp_name"]

    print("\n" + "#" * 100)
    print(f"### TRAIN EXPERIMENT: {exp_name}")
    print("#" * 100)

    paths = make_output_paths(exp_name)
    train_roots = build_train_roots(exp_name)
    val_roots = build_val_roots(exp_name)

    val_samples = collect_samples_from_roots(val_roots, with_weight=False)
    val_case_ids = get_case_ids(val_samples)

    train_samples_all = collect_samples_from_roots(train_roots, with_weight=True)
    train_case_ids_all = get_case_ids(train_samples_all)

    overlap_case_ids = train_case_ids_all & val_case_ids
    filtered_train_samples = [
        s for s in train_samples_all
        if s["case_id"] not in overlap_case_ids
    ]

    print(f"val .pt files          : {len(val_samples)}")
    print(f"val case_ids           : {len(val_case_ids)}")
    print(f"train candidate .pt    : {len(train_samples_all)}")
    print(f"train candidate cases  : {len(train_case_ids_all)}")
    print(f"overlap case_ids       : {len(overlap_case_ids)}")
    print(f"filtered train .pt     : {len(filtered_train_samples)}")
    print(f"filtered train case_ids: {len(get_case_ids(filtered_train_samples))}")

    if len(filtered_train_samples) == 0:
        print(f"[SKIP] train samples が0件です: {exp_name}")
        return None

    if len(val_samples) == 0:
        print(f"[SKIP] val samples が0件です: {exp_name}")
        return None

    input_dim = infer_input_dim(filtered_train_samples)
    print(f"INPUT_DIM auto detected: {input_dim}")

    train_ds = FeatureDataset(filtered_train_samples, return_weight=True)
    val_ds = FeatureDataset(val_samples)

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )

    results = []
    all_histories = []

    overall_best_score = -1.0
    overall_best_seed = None
    overall_best_model_path = None

    for seed in range(1, NUM_SEEDS + 1):
        set_global_seed(seed)

        def worker_init_fn(worker_id):
            seed_worker(worker_id, seed)

        g_train = torch.Generator()
        g_train.manual_seed(seed)

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            worker_init_fn=worker_init_fn,
            generator=g_train,
        )

        model = MLPClassifier(in_dim=input_dim).to(DEVICE)

        criterion = nn.CrossEntropyLoss(reduction="none")
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        model, best_epoch, best_val_acc, best_val_loss, history = train_and_evaluate(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
            num_epochs=NUM_EPOCHS,
            patience=PATIENCE,
        )

        y_true_val, y_proba_val = collect_preds_from_loader(model, val_loader, DEVICE)

        (
            best_thr,
            best_acc,
            best_recall_0sens,
            best_spec_0sens,
            best_f1_0sens,
            best_score,
            best_precision_0sens,
            tn,
            fp,
            fn,
            tp,
        ) = get_metrics_with_best_cutoff(y_true_val, y_proba_val)

        model_path = paths["model_dir"] / f"seed_{seed:03d}_best_model.pth"

        torch.save({
            "exp": exp,
            "exp_name": exp_name,
            "seed": seed,
            "model_state_dict": model.state_dict(),
            "input_dim": input_dim,
            "best_epoch": best_epoch,
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "best_cutoff": best_thr,
            "best_score": best_score,
            "metric_name": "f1_0_sens_for_cutoff",
            "best_epoch_selection": "minimum_val_loss",
            "positive_class": "0_sens",
            "label_mapping": LABEL_MAPPING,
            "train_roots": train_roots,
            "train_root_weights": {str(Path(root)): float(weight) for root, weight in train_roots},
            "val_roots": val_roots,
        }, model_path)

        results.append({
            "exp_name": exp_name,
            "model_name": exp.get("model_name"),
            "feature_type": exp.get("feature_type"),
            "img_size": exp.get("img_size"),
            "input_dim": input_dim,
            "SEED": seed,
            "best_epoch": best_epoch,
            "best_val_acc_epoch": best_val_acc,
            "best_val_loss_epoch": best_val_loss,
            "best_cutoff": best_thr,
            "val_acc_at_best_cutoff": best_acc,
            "val_recall_0_sens": best_recall_0sens,
            "val_specificity_0_sens": best_spec_0sens,
            "val_precision_0_sens": best_precision_0sens,
            "val_f1_0_sens": best_f1_0sens,
            "val_score": best_score,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "model_path": str(model_path),
            "train_pts": len(filtered_train_samples),
            "val_pts": len(val_samples),
            "train_case_ids": len(get_case_ids(filtered_train_samples)),
            "val_case_ids": len(val_case_ids),
        })

        for h in history:
            all_histories.append({
                "exp_name": exp_name,
                "SEED": seed,
                **h,
            })

        print(
            f"[{exp_name}] SEED {seed:3d}: "
            f"best_epoch={best_epoch:2d}, "
            f"val_loss={best_val_loss:.4f}, "
            f"val_acc_epoch={best_val_acc:.4f}, "
            f"cutoff={best_thr:.2f}, "
            f"val_acc={best_acc:.4f}, "
            f"recall={best_recall_0sens:.4f}, "
            f"spec={best_spec_0sens:.4f}, "
            f"f1={best_f1_0sens:.4f}"
        )

        if best_score > overall_best_score:
            overall_best_score = best_score
            overall_best_seed = seed
            overall_best_model_path = model_path

    if overall_best_model_path is not None:
        shutil.copy2(overall_best_model_path, paths["best_model"])

    df_results = (
        pd.DataFrame(results)
        .sort_values("best_val_loss_epoch", ascending=True)
        .reset_index(drop=True)
    )

    topk_df = df_results.head(TOP_K_ENSEMBLE).copy()

    topk_seeds = topk_df["SEED"].tolist()
    topk_model_paths = topk_df["model_path"].tolist()

    print("\n===== Top5 seeds for ensemble =====")
    print(topk_df[[
        "SEED",
        "best_epoch",
        "best_cutoff",
        "best_val_loss_epoch",
        "best_val_acc_epoch",
        "val_recall_0_sens",
        "val_specificity_0_sens",
        "val_f1_0_sens",
        "val_score",
    ]].to_string(index=False))

    ensemble_models = []

    for _, row in topk_df.iterrows():
        ckpt = torch_load_safe(row["model_path"], map_location="cpu")

        model = MLPClassifier(in_dim=ckpt["input_dim"]).to(DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()

        ensemble_models.append(model)

    y_true_ens, y_proba_ens = collect_ensemble_preds_from_loader(
        ensemble_models,
        val_loader,
        DEVICE,
    )

    (
        ens_thr,
        ens_acc,
        ens_recall_0sens,
        ens_spec_0sens,
        ens_f1_0sens,
        ens_score,
        ens_precision_0sens,
        ens_tn,
        ens_fp,
        ens_fn,
        ens_tp,
    ) = get_metrics_with_best_cutoff(y_true_ens, y_proba_ens)

    ensemble_row = {
        "exp_name": exp_name,
        "model_name": exp.get("model_name"),
        "feature_type": exp.get("feature_type"),
        "img_size": exp.get("img_size"),
        "input_dim": input_dim,
        "ensemble_name": f"top{TOP_K_ENSEMBLE}_mean_prob",
        "topk_seeds": ",".join(map(str, topk_seeds)),
        "topk_model_paths": " | ".join(topk_model_paths),
        "ensemble_best_cutoff": ens_thr,
        "val_acc_at_best_cutoff": ens_acc,
        "val_recall_0_sens": ens_recall_0sens,
        "val_specificity_0_sens": ens_spec_0sens,
        "val_precision_0_sens": ens_precision_0sens,
        "val_f1_0_sens": ens_f1_0sens,
        "val_score": ens_score,
        "tn": ens_tn,
        "fp": ens_fp,
        "fn": ens_fn,
        "tp": ens_tp,
        "n_models": len(ensemble_models),
        "train_pts": len(filtered_train_samples),
        "val_pts": len(val_samples),
        "train_case_ids": len(get_case_ids(filtered_train_samples)),
        "val_case_ids": len(val_case_ids),
    }

    print("\n===== Top5 ensemble result on VAL =====")
    print(
        f"[{exp_name}] "
        f"topk_seeds={topk_seeds}, "
        f"cutoff={ens_thr:.2f}, "
        f"val_acc={ens_acc:.4f}, "
        f"recall={ens_recall_0sens:.4f}, "
        f"spec={ens_spec_0sens:.4f}, "
        f"f1={ens_f1_0sens:.4f}"
    )

    torch.save({
        "exp": exp,
        "exp_name": exp_name,
        "ensemble_name": f"top{TOP_K_ENSEMBLE}_mean_prob",
        "topk_seeds": topk_seeds,
        "topk_model_paths": topk_model_paths,
        "ensemble_best_cutoff": ens_thr,
        "val_score": ens_score,
        "metric_name": "topk_seed_selection_by_minimum_val_loss",
        "cutoff_metric_name": "f1_0_sens_for_cutoff",
        "best_epoch_selection": "minimum_val_loss",
        "positive_class": "0_sens",
        "label_mapping": LABEL_MAPPING,
        "input_dim": input_dim,
        "train_roots": train_roots,
        "train_root_weights": {str(Path(root)): float(weight) for root, weight in train_roots},
        "val_roots": val_roots,
    }, paths["ensemble_info"])

    df_history = pd.DataFrame(all_histories)
    df_ensemble = pd.DataFrame([ensemble_row])

    with pd.ExcelWriter(paths["train_xlsx"], engine="openpyxl") as writer:
        df_results.to_excel(writer, sheet_name="summary", index=False)
        df_history.to_excel(writer, sheet_name="history", index=False)
        df_ensemble.to_excel(writer, sheet_name="ensemble_top5", index=False)
        topk_df.to_excel(writer, sheet_name="top5_members", index=False)

    print("\n===== TRAIN saved =====")
    print(f"best seed      : {overall_best_seed}")
    print(f"best val score : {overall_best_score:.4f}")
    print(f"best model     : {paths['best_model']}")
    print(f"ensemble info  : {paths['ensemble_info']}")
    print(f"excel          : {paths['train_xlsx']}")

    return {
        "exp": exp,
        "exp_name": exp_name,
        "input_dim": input_dim,
        "paths": paths,
        "ensemble_row": ensemble_row,
        "summary_df": df_results,
    }


# =========================================================
# external評価
# =========================================================
def load_single_best_model(best_model_path, device):
    ckpt = torch_load_safe(best_model_path, map_location="cpu")

    model = MLPClassifier(in_dim=ckpt["input_dim"]).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    info = {
        "seed": ckpt.get("seed"),
        "input_dim": ckpt.get("input_dim"),
        "best_epoch": ckpt.get("best_epoch"),
        "best_val_acc": ckpt.get("best_val_acc"),
        "best_val_loss": ckpt.get("best_val_loss"),
        "best_cutoff": ckpt.get("best_cutoff", 0.5),
        "best_score": ckpt.get("best_score"),
        "metric_name": ckpt.get("metric_name", "f1_0_sens"),
        "positive_class": ckpt.get("positive_class", "0_sens"),
        "model_path": str(best_model_path),
    }

    return model, info


def load_ensemble_models(ensemble_info_path, device):
    info = torch_load_safe(ensemble_info_path, map_location="cpu")

    topk_seeds = info["topk_seeds"]
    topk_model_paths = info["topk_model_paths"]
    ensemble_best_cutoff = info["ensemble_best_cutoff"]

    models = []
    members = []

    for model_path in topk_model_paths:
        ckpt = torch_load_safe(model_path, map_location="cpu")

        model = MLPClassifier(in_dim=ckpt["input_dim"]).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()

        models.append(model)

        members.append({
            "seed": ckpt.get("seed"),
            "input_dim": ckpt.get("input_dim"),
            "best_epoch": ckpt.get("best_epoch"),
            "best_cutoff": ckpt.get("best_cutoff"),
            "best_score": ckpt.get("best_score"),
            "metric_name": ckpt.get("metric_name", "f1_0_sens"),
            "positive_class": ckpt.get("positive_class", "0_sens"),
            "model_path": str(model_path),
        })

    ens_info = {
        "topk_seeds": topk_seeds,
        "topk_model_paths": topk_model_paths,
        "ensemble_best_cutoff": ensemble_best_cutoff,
        "metric_name": info.get("metric_name", "f1_0_sens"),
        "positive_class": info.get("positive_class", "0_sens"),
        "members": members,
    }

    return models, ens_info


@torch.no_grad()
def collect_preds_with_meta_from_loader(model, loader, device):
    rows = []

    model.eval()

    for batch in loader:
        feats = batch["feat"].to(device)

        out = model(feats)
        proba_1 = torch.softmax(out, dim=1)[:, 1].cpu().numpy()

        labels = batch["label"].numpy()

        for i in range(len(labels)):
            rows.append({
                "path": batch["path"][i],
                "case_id": batch["case_id"][i],
                "class_name": batch["class_name"][i],
                "subfolder": batch["subfolder"][i],
                "label": int(labels[i]),
                "prob_1_0sens": float(proba_1[i]),
            })

    return pd.DataFrame(rows)


@torch.no_grad()
def collect_ensemble_preds_with_meta_from_loader(models, loader, device):
    rows = []

    for m in models:
        m.eval()

    for batch in loader:
        feats = batch["feat"].to(device)

        prob_list = []
        for model in models:
            out = model(feats)
            proba = torch.softmax(out, dim=1)[:, 1]
            prob_list.append(proba.unsqueeze(1))

        prob_stack = torch.cat(prob_list, dim=1)
        mean_prob = prob_stack.mean(dim=1).cpu().numpy()

        labels = batch["label"].numpy()

        for i in range(len(labels)):
            rows.append({
                "path": batch["path"][i],
                "case_id": batch["case_id"][i],
                "class_name": batch["class_name"][i],
                "subfolder": batch["subfolder"][i],
                "label": int(labels[i]),
                "prob_1_0sens": float(mean_prob[i]),
            })

    return pd.DataFrame(rows)


def apply_cutoff(df_pred, cutoff):
    df = df_pred.copy()
    df["pred"] = (df["prob_1_0sens"] >= cutoff).astype(int)
    return df


def aggregate_case_level(df_pred, cutoff=0.5, method="mean"):
    rows = []

    for case_id, g in df_pred.groupby("case_id"):
        label = int(g["label"].iloc[0])

        if method == "mean":
            prob = float(g["prob_1_0sens"].mean())
        elif method == "median":
            prob = float(g["prob_1_0sens"].median())
        else:
            raise ValueError("method must be 'mean' or 'median'")

        pred = int(prob >= cutoff)

        rows.append({
            "case_id": case_id,
            "class_name": g["class_name"].iloc[0],
            "subfolder": g["subfolder"].iloc[0],
            "label": label,
            "prob_1_0sens": prob,
            "pred": pred,
            "n_tiles": len(g),
        })

    return pd.DataFrame(rows)


def add_summary_row(summary_rows, exp_name, model_name, level_name, cutoff, metrics):
    summary_rows.append({
        "exp_name": exp_name,
        "model": model_name,
        "level": level_name,
        "cutoff": cutoff,
        "acc": metrics["acc"],
        "recall_0_sens": metrics["recall_0_sens"],
        "specificity_0_sens": metrics["specificity_0_sens"],
        "precision_0_sens": metrics["precision_0_sens"],
        "f1_0_sens": metrics["f1_0_sens"],
        "tn": metrics["tn"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "tp": metrics["tp"],
    })


def run_external_eval_for_experiment(exp, paths):
    exp_name = exp["exp_name"]

    print("\n" + "=" * 100)
    print(f"### EXTERNAL EVAL: {exp_name}")
    print("=" * 100)

    external_root = build_external_root(exp_name)

    if not external_root.exists():
        print(f"[SKIP] external root not found: {external_root}")
        return None

    ext_samples = collect_samples_from_root(external_root)

    if len(ext_samples) == 0:
        print(f"[SKIP] external samples が0件です: {exp_name}")
        return None

    print(f"external .pt files : {len(ext_samples)}")
    print(f"external case_ids  : {len(get_case_ids(ext_samples))}")

    df_ext_info = pd.DataFrame(ext_samples)

    ext_ds = FeatureDataset(ext_samples, return_meta=True)
    ext_loader = DataLoader(
        ext_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )

    summary_rows = []

    # -------------------------------------------------
    # single best
    # -------------------------------------------------
    single_model, single_info = load_single_best_model(paths["best_model"], DEVICE)

    df_single_tile = collect_preds_with_meta_from_loader(single_model, ext_loader, DEVICE)
    df_single_tile = apply_cutoff(df_single_tile, single_info["best_cutoff"])

    single_tile_metrics = calc_metrics(df_single_tile["label"], df_single_tile["pred"])

    df_single_case_mean = aggregate_case_level(
        df_single_tile,
        cutoff=single_info["best_cutoff"],
        method="mean",
    )
    single_case_mean_metrics = calc_metrics(
        df_single_case_mean["label"],
        df_single_case_mean["pred"],
    )

    df_single_case_median = aggregate_case_level(
        df_single_tile,
        cutoff=single_info["best_cutoff"],
        method="median",
    )
    single_case_median_metrics = calc_metrics(
        df_single_case_median["label"],
        df_single_case_median["pred"],
    )

    add_summary_row(summary_rows, exp_name, "single_best", "tile", single_info["best_cutoff"], single_tile_metrics)
    add_summary_row(summary_rows, exp_name, "single_best", "case_mean", single_info["best_cutoff"], single_case_mean_metrics)
    add_summary_row(summary_rows, exp_name, "single_best", "case_median", single_info["best_cutoff"], single_case_median_metrics)

    # -------------------------------------------------
    # top5 ensemble
    # -------------------------------------------------
    ens_models, ens_info = load_ensemble_models(paths["ensemble_info"], DEVICE)

    df_ens_tile = collect_ensemble_preds_with_meta_from_loader(ens_models, ext_loader, DEVICE)
    df_ens_tile = apply_cutoff(df_ens_tile, ens_info["ensemble_best_cutoff"])

    ens_tile_metrics = calc_metrics(df_ens_tile["label"], df_ens_tile["pred"])

    df_ens_case_mean = aggregate_case_level(
        df_ens_tile,
        cutoff=ens_info["ensemble_best_cutoff"],
        method="mean",
    )
    ens_case_mean_metrics = calc_metrics(
        df_ens_case_mean["label"],
        df_ens_case_mean["pred"],
    )

    df_ens_case_median = aggregate_case_level(
        df_ens_tile,
        cutoff=ens_info["ensemble_best_cutoff"],
        method="median",
    )
    ens_case_median_metrics = calc_metrics(
        df_ens_case_median["label"],
        df_ens_case_median["pred"],
    )

    add_summary_row(summary_rows, exp_name, "top5_ensemble", "tile", ens_info["ensemble_best_cutoff"], ens_tile_metrics)
    add_summary_row(summary_rows, exp_name, "top5_ensemble", "case_mean", ens_info["ensemble_best_cutoff"], ens_case_mean_metrics)
    add_summary_row(summary_rows, exp_name, "top5_ensemble", "case_median", ens_info["ensemble_best_cutoff"], ens_case_median_metrics)

    df_summary = pd.DataFrame(summary_rows)
    df_ensemble_members = pd.DataFrame(ens_info["members"])

    with pd.ExcelWriter(paths["external_xlsx"], engine="openpyxl") as writer:
        df_summary.to_excel(writer, sheet_name="summary", index=False)
        df_ext_info.to_excel(writer, sheet_name="external_samples", index=False)

        df_single_tile.to_excel(writer, sheet_name="single_tile", index=False)
        df_single_case_mean.to_excel(writer, sheet_name="single_case_mean", index=False)
        df_single_case_median.to_excel(writer, sheet_name="single_case_median", index=False)

        df_ens_tile.to_excel(writer, sheet_name="ens_tile", index=False)
        df_ens_case_mean.to_excel(writer, sheet_name="ens_case_mean", index=False)
        df_ens_case_median.to_excel(writer, sheet_name="ens_case_median", index=False)

        df_ensemble_members.to_excel(writer, sheet_name="ensemble_members", index=False)

    df_summary.to_csv(paths["external_csv"], index=False, encoding="utf-8-sig")

    print("\n===== EXTERNAL summary =====")
    print(df_summary.to_string(index=False))
    print(f"external excel : {paths['external_xlsx']}")
    print(f"external csv   : {paths['external_csv']}")

    return df_summary


# =========================================================
# 全実験まとめ実行
# =========================================================
def main():
    print(f"DEVICE: {DEVICE}")
    print(f"folder_name: {folder_name}")
    print(f"NUM_SEEDS: {NUM_SEEDS}")

    all_train_ensemble_rows = []
    all_external_summaries = []

    for exp in EXPERIMENTS:
        exp_name = exp["exp_name"]

        try:
            train_result = run_training_for_experiment(exp)

            if train_result is None:
                continue

            all_train_ensemble_rows.append(train_result["ensemble_row"])

            ext_summary = run_external_eval_for_experiment(
                exp=exp,
                paths=train_result["paths"],
            )

            if ext_summary is not None:
                all_external_summaries.append(ext_summary)

        except Exception as e:
            print("\n" + "!" * 100)
            print(f"[ERROR] experiment failed: {exp_name}")
            print(f"{type(e).__name__}: {e}")
            print("次のexperimentへ進みます。")
            print("!" * 100)

            continue

        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # -------------------------------------------------
    # 全experiment横断summary保存
    # -------------------------------------------------
    summary_dir = RESULT_BASE / folder_name
    summary_dir.mkdir(parents=True, exist_ok=True)

    summary_xlsx = summary_dir / "ALL_EXPERIMENTS_SUMMARY.xlsx"

    df_train_ens = pd.DataFrame(all_train_ensemble_rows)

    if len(all_external_summaries) > 0:
        df_external_all = pd.concat(all_external_summaries, ignore_index=True)
    else:
        df_external_all = pd.DataFrame()

    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        pd.DataFrame(EXPERIMENTS).to_excel(writer, sheet_name="experiments", index=False)
        df_train_ens.to_excel(writer, sheet_name="val_ensemble_summary", index=False)
        df_external_all.to_excel(writer, sheet_name="external_summary", index=False)

    print("\n" + "#" * 100)
    print("ALL DONE")
    print(f"summary excel: {summary_xlsx}")
    print("#" * 100)


if __name__ == "__main__":
    main()